# 第46课 · 人耳是台对数机器——Mel 频率尺度与三角滤波器组（mel filterbank）

**目标**：实现 Hz↔Mel 转换和三角滤波器组（triangular filter bank），理解为什么 Mel 特征比线性频谱更适合 ASR。

🔗 Aurora 连接：`aurora.audio.mel.hz_to_mel()` 和 `aurora.audio.mel.mel_filterbank()`

← **上一课**　[L45 · 声谱图（spectrogram）](L45_spectrogram.ipynb)

> 上节课学习了 **声谱图（spectrogram）**：从 |STFT|² 功率谱到 dB 热力图，给声音拍一张时频 X 光片。  
> 本课将探讨 **Mel 频率尺度**。

## 本课剧情：为什么钢琴键盘不是等间距？

钢琴的 88 个键，从低音到高音频率跨越 27 Hz 到 4186 Hz。但你注意到吗——低音区相邻键的频率差只有几赫兹，高音区相邻键却差几百赫兹。这不是设计错误，这是人耳的秘密。

**人耳是对数感知器**：你能分辨 100 Hz vs 110 Hz（差 10 Hz），却几乎听不出 3000 Hz vs 3010 Hz（同样差 10 Hz）。对人耳来说，"音高差一个八度"意味着频率翻倍——这是乘法关系，不是加法。

Mel 尺度（1937 年）把人耳的对数感知数学化：

$$\text{mel}(f) = 2595 \cdot \log_{10}\!\left(1 + \frac{f}{700}\right)$$

**关键数值**：
- 1000 Hz → 1000 mel（标定点）
- 2000 Hz → 1521 mel（频率增加 1000 Hz，mel 只加 521）
- 4000 Hz → 2146 mel（频率再增加 2000 Hz，mel 也只加 625）

看到规律了吗：Hz 的增量翻了一倍（1000 → 2000），mel 的增量却几乎没变。频率越高，同样多的 Hz 在 mel 轴上被压得越扁——这正是对数压缩（log compression），跟人耳"高频不敏感"的感知一致。

**Mel 滤波器组（filterbank）**：在 Mel 域均匀排列 N 个三角形滤波器，投影到线性 Hz 轴后低频密、高频疏——模拟耳蜗的物理结构。这是 MFCC 特征的第一步。

本节任务：实现 `hz_to_mel(f)` 和 `mel_filterbank(n_mels, n_fft, sr)`。

## 视觉桥梁：钢琴键 → 频率 → Mel → 耳蜗

为什么"低频分得细、高频分得粗"？顺着这条链看一眼就懂（纯直觉，公式在下一节）：

```
钢琴键盘（琴键等宽、听感等距）
   │  每高一个八度 = 物理频率翻倍（×2，不是 +固定值）
   ▼
物理频率   27 → 55 → 110 → 220 → 440 → 880 → 1760 Hz   （越往右，八度间隔越宽）
   │  hz_to_mel()：把"翻倍"这种乘法关系拉直成加法
   ▼
Mel 轴     |--|--|--|--|--|--|   八度在这里变成等间隔
   │  均匀切成 N 段，再反投影回 Hz
   ▼
耳蜗基底膜  低频区排布密 ‖‖‖‖‖ …… 高频区排布疏 ‖   ‖     ‖
```

一句话：人耳按**倍数**（对数）感知音高，Mel 轴就是把这种"倍数感"拉成"等距"。所以回到 Hz 上看，低频滤波器又窄又密、高频滤波器又宽又疏——正好贴合耳蜗基底膜的物理排布。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

## 1. Mel 标度公式

Mel 标度由 Stevens、Volkmann、Newman（1937）提出，常用近似公式为：

```
f_mel = 2595 * log10(1 + f_hz / 700)
```

700 Hz 的音调对应约 781 Mel。注意：
- 低频区间（0–1000 Hz）：每 100 Hz 对应约 100+ Mel，分辨率高
- 高频区间（3000–4000 Hz）：每 100 Hz 对应约 30 Mel，分辨率低

逆变换（Mel → Hz）：
```
f_hz = 700 * (10^(f_mel / 2595) - 1)
```

In [ ]:
# 演示 Mel 标度的对数特性
freqs = np.array([100, 200, 500, 1000, 2000, 4000, 8000])
mels_approx = 2595 * np.log10(1 + freqs / 700)

print(f"{'Hz':>8}  {'Mel':>8}  {'Delta_Mel':>10}")
for i, (f, m) in enumerate(zip(freqs, mels_approx)):
    delta = f"{mels_approx[i] - mels_approx[i-1]:.1f}" if i > 0 else "—"
    print(f"{f:>8}  {m:>8.1f}  {delta:>10}")

## 2. 三角滤波器组的构造

构造步骤：
1. 在 Mel 域从 0 到 `hz_to_mel(sr/2)` 均匀取 `n_mels + 2` 个中心点（含左右边界）
2. 将这 `n_mels + 2` 个 Mel 值转回 Hz，得到非均匀间隔的中心频率
3. 把每个 Hz 中心频率映射到最近的 FFT bin 索引
4. 第 `m` 个三角滤波器：在 `[center[m-1], center[m]]` 线性升至 1，在 `[center[m], center[m+1]]` 线性降至 0

每个滤波器的形状（在 Hz 域）是不对称三角形——低频滤波器窄，高频滤波器宽，天然匹配人耳分辨率。

In [ ]:
# 可视化：Mel 域均匀 → Hz 域非均匀
sr = 16000
n_mels = 10
mel_max = 2595 * np.log10(1 + (sr / 2) / 700)
mel_points = np.linspace(0, mel_max, n_mels + 2)
hz_points = 700 * (10 ** (mel_points / 2595) - 1)

fig, axes = plt.subplots(1, 2, figsize=(10, 3))
axes[0].plot(mel_points, np.zeros(n_mels+2), 'o')
axes[0].set_title('Mel 域：均匀间隔')
axes[0].set_xlabel('Mel')
axes[1].plot(hz_points, np.zeros(n_mels+2), 'o')
axes[1].set_title('Hz 域：对数间隔（低密高疏）')
axes[1].set_xlabel('Hz')
plt.tight_layout()
plt.show()

## 3. Filterbank 矩阵与功率谱（power spectrum）相乘

`mel_filterbank()` 返回 shape `(n_mels, n_fft//2+1)` 的矩阵，每行是一个三角滤波器在 FFT bin 轴上的权重。

```
fb = mel_filterbank(40, 512, 16000)   # shape (40, 257)
power_spec = np.abs(X)**2             # shape (257,)，单帧功率谱
mel_energy = fb @ power_spec          # shape (40,)，Mel 能量
```

`@` 一次矩阵乘法完成 40 个三角积分。对 `mel_energy` 取 log 即得 log-Mel 特征，再做 DCT 即得 MFCC（L49-L50）。

In [ ]:
# 演示矩阵乘法：随机功率谱 -> Mel 能量
# （真实 filterbank 在编码任务后实现）
np.random.seed(42)
n_fft = 512
fake_power = np.random.rand(n_fft // 2 + 1)  # shape (257,)
# 占位：用全 1 filterbank 演示形状
fb_placeholder = np.ones((40, n_fft // 2 + 1)) / (n_fft // 2 + 1)
mel_e = fb_placeholder @ fake_power
print(f"功率谱 shape: {fake_power.shape}")
print(f"filterbank shape: {fb_placeholder.shape}")
print(f"Mel 能量 shape: {mel_e.shape}")

## 4. ✏️ 实现 `hz_to_mel(f)`

**公式直接套用**：$\text{mel}(f) = 2595 \cdot \log_{10}(1 + f/700)$

| 实现提示 | 代码 |
|---|---|
| 单个频率 | `2595 * np.log10(1 + f / 700)` |
| 数组输入 | `np.log10` 广播，直接适用 |

**验收数值**（手算对照）：
- `hz_to_mel(0)` = 0.0（0 Hz = 0 mel）
- `hz_to_mel(700)` ≈ 781.2 mel
- `hz_to_mel(1000)` ≈ 1000.0 mel（标定点）
- `hz_to_mel(4000)` ≈ 2146.1 mel

> 逆变换：`mel_to_hz(m) = 700 * (10^(m/2595) - 1)`

In [ ]:
def hz_to_mel(f):
    # ✏️ TODO: 套用公式 2595 * log10(1 + f/700)
    # 提示：用 np.log10，支持标量和数组
    raise NotImplementedError("TODO: 返回 2595 * np.log10(1 + f / 700)")

In [ ]:
assert abs(hz_to_mel(700) - 781.2) < 0.5, f"hz_to_mel(700) = {hz_to_mel(700):.2f}"
assert hz_to_mel(0) == 0.0, "hz_to_mel(0) 应为 0"
result = hz_to_mel(np.array([0, 700, 8000]))
assert result.shape == (3,), "数组输入应返回同 shape"
print(f"hz_to_mel(700) = {hz_to_mel(700):.2f}  ✅")
print(f"hz_to_mel([0, 700, 8000]) = {hz_to_mel(np.array([0, 700, 8000]))}  ✅")

## 5. ✏️ 实现 `mel_filterbank(n_mels, n_fft, sr)`

**推理路线**：
1. Mel 域取 `n_mels + 2` 个均匀点（含 0 和 `hz_to_mel(sr/2)` 两端），用 `np.linspace`
2. 将所有 Mel 点转回 Hz：`700 * (10**(mel_points / 2595) - 1)`
3. 把 Hz 映射到 FFT bin 索引：`np.floor(hz_points / (sr / n_fft) * ... )`——精确公式为 `np.floor((n_fft + 1) * hz_points / sr).astype(int)`
4. 对每个滤波器 `m`（0 到 n_mels-1），在 `[bin[m], bin[m+1], bin[m+2]]` 区间构造上升/下降斜坡：
   - 上升段：`(k - bin[m]) / (bin[m+1] - bin[m])` for k in `[bin[m], bin[m+1]]`
   - 下降段：`(bin[m+2] - k) / (bin[m+2] - bin[m+1])` for k in `[bin[m+1], bin[m+2]]`

**参考输入输出**：
- `mel_filterbank(40, 512, 16000).shape` → `(40, 257)`
- 所有元素 `>= 0`，每行峰值为 1
- `mel_filterbank(40, 512, 16000)[0, :10]`：前几个 bin 有非零权重（第 0 个滤波器覆盖低频）

> 💡 需要提示？完成练习后可参考 `solutions/` 目录中的参考实现。


In [ ]:
# ✏️ 步骤 1-3：Mel 频率转换——先把 Hz 映射到 Mel 域，再反算回 Hz，再换算成 FFT bin 索引
def mel_filterbank_steps123(n_mels, n_fft, sr):
    """返回 (mel_min, mel_max, mel_points, hz_points, bin_idx) 供调试核对。"""
    mel_min = 0.0
    # ✏️ TODO 1: mel_max = hz_to_mel(sr / 2)
    # ✏️ TODO 2: mel_points = np.linspace(mel_min, mel_max, n_mels + 2)
    # ✏️ TODO 3a: hz_points = 700 * (10 ** (mel_points / 2595) - 1)
    # ✏️ TODO 3b: bin_idx = np.floor((n_fft + 1) * hz_points / sr).astype(int)
    raise NotImplementedError("TODO 1-3: 完成 mel_max / mel_points / hz_points / bin_idx 后删除此行")
    return mel_min, mel_max, mel_points, hz_points, bin_idx

try:
    _, _, mpts, hpts, bidx = mel_filterbank_steps123(26, 512, 16000)
    assert mpts is not None, '⬜ mel_points 未实现'
    assert len(mpts) == 28, f'mel_points 长度应为 28（n_mels+2），实际 {len(mpts)}'
    assert abs(hpts[0]) < 1.0, f'最低 Hz 点应接近 0，实际 {hpts[0]:.1f}'
    assert abs(hpts[-1] - 8000) < 10, f'最高 Hz 点应接近 8000，实际 {hpts[-1]:.1f}'
    print(f'✅ 步骤 1-3 通过：bin_idx 首尾 = {bidx[0]}, {bidx[-1]}')
except NotImplementedError:
    print('⬜ mel_filterbank_steps123 未实现')


In [ ]:
# ✏️ 步骤 4：三角滤波器矩阵——用步骤 1-3 的 bin_idx 填充 fb
def mel_filterbank(n_mels, n_fft, sr):
    """完整 mel 滤波器组，返回形状 (n_mels, n_fft//2+1) 的三角矩阵。"""
    mel_min = 0.0
    mel_max = hz_to_mel(sr / 2)
    mel_points = np.linspace(mel_min, mel_max, n_mels + 2)
    hz_points = 700 * (10 ** (mel_points / 2595) - 1)
    bin_idx = np.floor((n_fft + 1) * hz_points / sr).astype(int)

    n_bins = n_fft // 2 + 1
    fb = np.zeros((n_mels, n_bins))
    # ✏️ TODO: 对第 m 个滤波器，lo=bin_idx[m]，ctr=bin_idx[m+1]，hi=bin_idx[m+2]
    #   上升段 [lo, ctr)：fb[m, k] = (k - lo) / (ctr - lo)
    #   下降段 [ctr, hi)：fb[m, k] = (hi - k) / (hi - ctr)
    return fb

try:
    fb = mel_filterbank(26, 512, 16000)
    assert fb.shape == (26, 257), f'形状应为 (26, 257)，实际 {fb.shape}'
    assert (fb >= 0).all(), '滤波器值应非负'
    assert fb.max() > 0, 'fb 全为 0，步骤 4 尚未实现'
    print(f'✅ mel_filterbank 形状正确：{fb.shape}，最大值 {fb.max():.4f}')
except NotImplementedError:
    print('⬜ mel_filterbank 步骤 4 未实现')


In [ ]:
fb = mel_filterbank(40, 512, 16000)
assert fb.shape == (40, 257), f"shape 错误: {fb.shape}"
assert np.all(fb >= 0), "滤波器权重不应为负"
assert np.allclose(fb.max(axis=1), 1.0), "每个滤波器峰值应为 1"
print(f"filterbank shape: {fb.shape}  ✅")
print(f"所有权重 >= 0: {np.all(fb >= 0)}  ✅")
print(f"各行最大值均为 1: {np.allclose(fb.max(axis=1), 1.0)}  ✅")

## 6. 参数实验：可视化滤波器组

修改参数观察效果：
- `n_mels=40`（标准 ASR）vs `n_mels=80`（高分辨率）：滤波器数量翻倍，高频细节更丰富
- `sr=16000` vs `sr=8000`：采样率（sample rate，sr）减半时最高覆盖频率从 8000 Hz 降至 4000 Hz，滤波器更集中在低频
- 图1（Hz 线性轴）：低频滤波器窄且密集，高频滤波器宽且稀疏——体现对数特性
- 图2（Mel 轴）：所有滤波器宽度相等，均匀分布

In [ ]:
fb40 = mel_filterbank(40, 512, 16000)
n_bins = 512 // 2 + 1
freq_axis = np.linspace(0, 16000 / 2, n_bins)
mel_axis = hz_to_mel(freq_axis)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# 图1：Hz 线性轴
for m in range(40):
    axes[0].plot(freq_axis, fb40[m], alpha=0.5, linewidth=0.8)
axes[0].set_title('40 个 Mel 滤波器（Hz 线性轴）')
axes[0].set_xlabel('频率 (Hz)')
axes[0].set_ylabel('权重')
axes[0].set_xlim(0, 8000)

# 图2：Mel 轴
for m in range(40):
    axes[1].plot(mel_axis, fb40[m], alpha=0.5, linewidth=0.8)
axes[1].set_title('40 个 Mel 滤波器（Mel 轴）')
axes[1].set_xlabel('频率 (Mel)')
axes[1].set_ylabel('权重')

plt.suptitle('n_mels=40, n_fft=512, sr=16000', fontsize=12)
plt.tight_layout()
plt.show()
print("左图低频密/高频疏 → 右图均匀分布，体现 Mel 标度的对数特性")

## 3a. 手绘练习：三角滤波器三锚点

在纸上画一个三角滤波器，标出 left / center / right 三个锚点，再写出上升段和下降段的斜率。
如果卡住，回想 `hz_to_mel` 只是坐标变换，真正的形状来自三个相邻中心点。

## 本课收束

本课实现了 `hz_to_mel()` 和 `mel_filterbank()`，前者把线性 Hz 映射到感知对数 Mel 标度，后者构造 shape `(n_mels, n_fft//2+1)` 的三角权重矩阵。将该矩阵乘以 STFT 功率谱，一步得到 `(n_mels,)` 的 Mel 能量向量，与 `aurora.audio.mel` 模块中的实现等价。下一课（L47）将把滤波器组接入完整流水线——STFT → 功率谱 → Mel 滤波 → log 压缩，亲手实现 `log_mel_spectrogram` 并与 aurora 参考输出对齐。

## ✏️ 白板挑战：Mel 尺度手算（目标 10 分钟）

盖上屏幕，纸上作答：

**问 1**：手算以下 Hz → Mel：
- 100 Hz → ? mel
- 1000 Hz → ? mel（标定点）
- 8000 Hz → ? mel

**问 2**：`mel_to_hz(m)` 的反变换公式是什么？  
（提示：从 mel = 2595·log₁₀(1+f/700) 反解 f）  
验证：mel_to_hz(hz_to_mel(440)) ≈ 440

**问 3**：`n_mels=40, n_fft=512, sr=16000`：
- 最高 Mel 频率 = hz_to_mel(sr/2) = hz_to_mel(8000) = ?
- Mel 域等间距取 42 个点（含两端），相邻间距 = ? mel

**问 4**：三角滤波器组矩阵 `fb` 的 shape 是什么？  
（n_mels=40, n_fft=512 → n_bins=n_fft//2+1=?）

**问 5**：为什么 filterbank 矩阵与功率谱相乘时用**矩阵乘法**而不是逐元素乘法？  
（提示：功率谱 shape=(n_bins,)，fb shape=(n_mels, n_bins)，如何得到 (n_mels,)?）

推导完成后运行下面格对答案。

In [ ]:
# ✏️ 对答案格
import numpy as np

# 问1：手算 Hz → Mel
def _ref_hz_to_mel(f):
    return 2595 * np.log10(1 + f / 700)

mel_100  = _ref_hz_to_mel(100)
mel_1000 = _ref_hz_to_mel(1000)
mel_8000 = _ref_hz_to_mel(8000)

assert abs(mel_100 - 150.5) < 1.0
assert abs(mel_1000 - 999.9) < 1.0
assert abs(mel_8000 - 2840.0) < 5.0
print(f"Q1 ✅  100→{mel_100:.1f}mel，1000→{mel_1000:.1f}mel，8000→{mel_8000:.1f}mel")

# 问2：mel_to_hz 反变换验证
def _ref_mel_to_hz(m):
    return 700 * (10 ** (m / 2595) - 1)

f_roundtrip = _ref_mel_to_hz(_ref_hz_to_mel(440))
assert abs(f_roundtrip - 440) < 1e-8
print(f"Q2 ✅  mel_to_hz(hz_to_mel(440)) = {f_roundtrip:.10f} ≈ 440 Hz")

try:
    from aurora.audio.mel import mel_to_hz as aurora_mel_to_hz
    print(f"    Aurora mel_to_hz(hz_to_mel(440)) = {aurora_mel_to_hz(_ref_hz_to_mel(440)):.6f}")
except (ImportError, AttributeError):
    pass

# 问3：n_mels=40, n_fft=512, sr=16000
n_mels, n_fft, sr = 40, 512, 16000
mel_max = _ref_hz_to_mel(sr / 2)
n_center = n_mels + 2
mel_spacing = mel_max / (n_center - 1)
assert abs(mel_max - 2840.0) < 5.0
print(f"Q3 ✅  hz_to_mel(8000)={mel_max:.1f} mel，42点间距={mel_spacing:.1f} mel")

# 问4：filterbank shape
n_bins = n_fft // 2 + 1
assert n_bins == 257
try:
    fb = mel_filterbank(n_mels, n_fft, sr)
    assert fb.shape == (n_mels, n_bins), f"shape错误: {fb.shape}"
    print(f"Q4 ✅  fb.shape = {fb.shape}（{n_mels}个滤波器 × {n_bins}个FFT bin）")
except (NotImplementedError, NameError):
    print(f"Q4 ✅  n_bins={n_bins}，fb.shape=({n_mels},{n_bins})（mel_filterbank 待实现）")

# 问5：矩阵乘法 vs 逐元素
# power_spectrum shape=(n_bins,)，fb shape=(n_mels, n_bins)
# mel_energy = fb @ power  → (n_mels, n_bins) @ (n_bins,) = (n_mels,)
power = np.ones(n_bins)
fb_ref = np.ones((n_mels, n_bins)) / n_bins  # dummy uniform filterbank
mel_energy = fb_ref @ power
assert mel_energy.shape == (n_mels,)
print(f"Q5 ✅  fb({n_mels},{n_bins}) @ power({n_bins},) = mel_energy({n_mels},)（矩阵乘法降维）")
print("\n🎉 Mel 尺度白板挑战通过！")

In [ ]:
# ✏️ 本课自评
l46_review = {
    "mel_formula":             None,  # 记住 mel=2595·log₁₀(1+f/700)和反变换？True/False
    "hz_to_mel_impl":          None,  # hz_to_mel 实现并通过 hz_to_mel(700)≈781 断言？True/False
    "filterbank_impl":         None,  # mel_filterbank shape=(n_mels,n_fft//2+1) 通过？True/False
    "log_perception":          None,  # 理解人耳对数感知→Mel等间距≈耳蜗均匀分布？True/False
    "whiteboard_passed":       None,  # 白板挑战纸上推导完成？True/False
}

unfilled = [k for k, v in l46_review.items() if v is None]
assert not unfilled, f'还未填写：{unfilled}'
weak = [k for k, v in l46_review.items() if v is False]
if weak:
    print(f'⚠️  需要加强：{weak}')
else:
    print('✅ L46 全部通关！进入 L47：亲手写 Mel 滤波器')

---

→ **下一课**　[L47 · 亲手搭建 log-Mel 流水线](L47_mel_implement.ipynb)

> 下节课将学习 **亲手搭建 log-Mel 流水线**：log_mel_spectrogram 从公式到 NumPy，与仓库输出对齐。